# ECE284 Project Part1 — W4A4 VGG16

目标：13 个卷积均为 W4A4；`features[27]` 为 8→8、后无 BN；CIFAR-10 测试准确率 >90%；整数卷积加 ReLU 与下一层真实 prehook 输入的最大绝对误差 <1e-3，争取 <1e-5。

**当前 notebook 没有训练结果。请在 DSMLP GPU 环境运行。包内 qa 是代码检查，不是达标训练证据。**

保留 notebook 与 `models.py`、`train.py`、`verify_export.py` 在同一目录。

In [ ]:
from pathlib import Path
import sys, os, subprocess, json, csv
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "train.py").is_file():
    candidate = PROJECT_ROOT / "ECE284_Part1"
    if candidate.is_dir():
        PROJECT_ROOT = candidate
    else:
        raise FileNotFoundError("请在解压后的 ECE284_Part1 文件夹打开 notebook，保留同目录的 .py 文件。")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import torch
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("当前无 CUDA。快速检查可在 CPU 运行；正式训练建议切换 DSMLP GPU 环境。")


## 1. 先检查模型和量化配置

首层输入是标准化 RGB，可能含负值，采用 signed 4-bit；其他卷积的 ReLU 输入采用 unsigned 4-bit。分类 Linear 和 BN 参数保留浮点。选定的硬件层没有 bias，后面只有 ReLU，因此能够直接比较整数恢复值与下一卷积的输入。

In [ ]:
from models import build_model, QuantConv2d
model = build_model(quantize=True)
print(model.features[24:31])
print(json.dumps(model.architecture_config(), indent=2))
assert model.features[27].in_channels == model.features[27].out_channels == 8
assert isinstance(model.features[28], torch.nn.ReLU)
assert isinstance(model.features[29], QuantConv2d)
assert len([m for m in model.modules() if isinstance(m, QuantConv2d)]) == 13
# 释放本单元模型；训练脚本会创建自己的模型。
del model


## 2. 运行独立检查

这一步无需 CIFAR-10，检查量化梯度、真实下一层 prehook、int64 卷积和导出的二进制布局。完成这一步不代表准确率达到 90%。

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## 3. 配置正式训练

第一次保持 `ACTION="train"`；中断后保留原配置，将它改为 `"resume"`；只重新测已有模型则设为 `"evaluate"`。

默认 200 epoch，从头执行 QAT。第一次自动下载 CIFAR-10。`best.pth` 保存全测试集准确率最高的 QAT 模型。此流程沿用课程参考的选模方式，测试集不属于研究意义上未触碰的 holdout。

In [ ]:
ACTION = "train"          # "train" / "resume" / "evaluate"
DEVICE = "auto"           # 自动优先 CUDA，其次 MPS，最后 CPU
DATA_ROOT = PROJECT_ROOT / "data"
RUN_DIR = PROJECT_ROOT / "runs" / "part1"
QAT_EPOCHS = 200
BATCH_SIZE = 128
WORKERS = 2
SEED = 284
DETERMINISTIC = False
ALLOW_CPU_TRAINING = False
print("Action:", ACTION, "Output:", RUN_DIR)


## 4. 正式训练、完整测试及导出

这一单元耗时最长。每个 epoch 打印 train/test accuracy 并保存 `last.pth`，支持中断恢复。

完成后自动重载 `best.pth`，重新测试 10,000 张 CIFAR-10 测试图像，再对固定 test index 0 进行整数重算和向量导出。**只有运行产生的数据才是最终结果。**

In [ ]:
from train import choose_device
actual_device = choose_device(DEVICE)
if actual_device.type == "cpu" and not ALLOW_CPU_TRAINING:
    raise RuntimeError("当前只有 CPU。请切换 DSMLP GPU 环境后运行；若明确要 CPU 长时间训练，再将 ALLOW_CPU_TRAINING=True。")
if ACTION not in {"train", "resume", "evaluate"}:
    raise ValueError("ACTION 必须为 train、resume 或 evaluate")
command = [sys.executable, "train.py", "--device", DEVICE,
           "--data-root", str(DATA_ROOT), "--output", str(RUN_DIR),
           "--qat-epochs", str(QAT_EPOCHS), "--batch-size", str(BATCH_SIZE),
           "--workers", str(WORKERS), "--seed", str(SEED)]
if DETERMINISTIC:
    command += ["--deterministic"]
if ACTION == "resume":
    command += ["--resume", str(RUN_DIR / "last.pth")]
elif ACTION == "evaluate":
    command += ["--evaluate", str(RUN_DIR / "best.pth")]
print("Running:", " ".join(command))
completed = subprocess.run(command)
if completed.returncode != 0:
    if completed.returncode == 2 and (RUN_DIR / "results.json").is_file():
        print("运行结束，但 accuracy 或误差条件尚未全部通过。请查看下一单元的实际结果。")
    else:
        raise RuntimeError(f"运行错误，返回码 {completed.returncode}；请检查上方具体报错。")


## 5. 看最终结果，不用 signed error sum 判定

正式成功条件是 test accuracy **>90%** 和最大绝对误差 **<1e-3**。正文更严格的 **<1e-5** 也会单独报告。Float32 累加可能让更严格的条件失败，即使整数与硬件验证正确；请如实保留两项结果。

In [ ]:
results_path = RUN_DIR / "results.json"
if not results_path.is_file():
    raise FileNotFoundError("还没有正式训练/测试结果。先完成上一单元。")
results = json.loads(results_path.read_text())
assert results["evidence_type"] == "trained_model"
print(json.dumps(results, indent=2, ensure_ascii=False))
print("\n报告可填写的实际数值：")
print(f"Accuracy: {results['test_accuracy_pct']:.4f}% ({results['correct']}/{results['total']})")
print(f"Max absolute reconstruction error: {results['max_abs_error']:.10g}")
print("Formal success:", results["status"])
print("Strict <1e-5:", results["error_lt_1e_5"])


## 6. 训练曲线

用于说明训练是否收敛以及 QAT 测试准确率是否达到目标。曲线来自 `history.csv`，没有预填结果。

In [ ]:
import matplotlib.pyplot as plt
history_path = RUN_DIR / "history.csv"
with history_path.open() as f:
    records = list(csv.DictReader(f))
qat = [r for r in records if r["phase"] == "qat"]
if not qat:
    raise RuntimeError("没有 QAT epoch 记录")
x = [int(r["epoch"]) for r in qat]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(x, [float(r["train_accuracy_pct"]) for r in qat], label="Train")
axes[0].plot(x, [float(r["test_accuracy_pct"]) for r in qat], label="CIFAR-10 test")
axes[0].axhline(90, color="gray", linestyle="--", label="90% target")
axes[0].set(xlabel="QAT epoch", ylabel="Accuracy (%)")
axes[0].legend()
axes[1].plot(x, [float(r["train_loss"]) for r in qat], label="Train loss incl. alpha regularization")
axes[1].plot(x, [float(r["test_loss"]) for r in qat], label="Test CE loss")
axes[1].set(xlabel="QAT epoch", ylabel="Loss")
axes[1].legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "training_curves.png", dpi=180)
plt.show()


## 7. 与 Part4 交接

交接整个 `runs/part1/vectors/`，保持文件相对路径。`input.txt` 是 padded activation，`weight.txt` 是合并权重，`psum.txt` 是 **整数域 ReLU 后期望值**。旧 testbench 使用 `activation/`、`weights/`、`output/` 和 `acc_address.txt`。

文件位序、scales、样本索引和模型 SHA256 都在 metadata 中。硬件验证通过后，组内 RTL 负责人还需完成 SFU 结果写回 SRAM 的要求；本 notebook 只负责 Part1 及数据接口。

In [ ]:
vector_dir = RUN_DIR / "vectors"
metadata = json.loads((vector_dir / "metadata.json").read_text())
print("Model fingerprint:", metadata["model_sha256"])
print("Dataset provenance:", metadata["provenance"])
print(json.dumps(metadata["vectors"], indent=2, ensure_ascii=False))
print("交接目录:", vector_dir)


## 8. 报告 Part1 应写什么

用你刚运行出的结果填写：

1. 模型修改：目标层位置、前后通道衔接、8→8、去掉该层后的 BN。
2. 量化方法：W4A4、首层 signed 输入、其他层 unsigned 输入、权重量化前的标准化、learned clipping。
3. 训练设置和实际完整测试准确率，附训练曲线。
4. 验证方法：int64 卷积与 ReLU，恢复到模型数值域，比较下一卷积真实 prehook 输入。
5. 实际最大绝对误差，以及 `<1e-5` 和 `<1e-3` 两个判断。
6. 推论：模型保留的分类能力如何，以及整数计算与量化模型是否数值一致。

保留 `best.pth`、代码、notebook、日志、results 和 vectors，便于组内整合。提交格式以课程最终说明为准。
正式测试结束后，脚本还会生成 `runs/part1/part1_report.txt`，包含实际数值的英文初稿。
